# KG-Flan: End-to-End Inference Demo

**KG-Flan** — Cost-Efficient Knowledge Graph Reasoning with Trained Relation Scoring

This notebook walks through the full pipeline:
1. Load the MLP Relation Scorer
2. Score candidate relations for a sample claim
3. Build an evidence subgraph
4. Run Flan-T5-XL inference

> Run on **Kaggle** (free T4 GPU) or locally on CPU (slower).

In [ ]:
# Install dependencies (only needed first time)
!pip install sentence-transformers transformers torch -q

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import torch
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load the MLP Relation Scorer

In [ ]:
from src.model import MLPRelationScorer

# Instantiate fresh scorer (or load from checkpoint if available)
CKPT = '../checkpoints/factkg_scorer.pt'

if os.path.exists(CKPT):
    from src.model import load_scorer
    scorer = load_scorer(CKPT, str(device))
    print('Loaded scorer from checkpoint.')
else:
    scorer = MLPRelationScorer().to(device)
    print('Using fresh (untrained) scorer — run train.py first for real results.')

encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Encoder loaded. Scorer params: {scorer.count_parameters():,}')

## 2. Score candidate relations for a sample claim

In [ ]:
from src.utils import score_candidates

claim = 'Christopher Nolan directed The Dark Knight which was released in 2008.'
candidates = [
    'director', 'starring', 'release_year', 'birthPlace',
    'spouse', 'award', 'genre', 'cinematography', 'distributor', 'budget'
]

scored = score_candidates(claim, candidates, scorer, encoder, str(device))

print(f'Claim: {claim}\n')
print(f'{"Relation":<25} {"Score":>7}')
print('-' * 35)
for rel, score in scored:
    print(f'{rel:<25} {score:>7.4f}')

## 3. Build an evidence subgraph (demo with mock KB)

In [ ]:
from src.utils import build_evidence_graph, triples_to_string

# Mock knowledge graph (replace with real DBpedia/MetaQA KB)
mock_kg = {
    'The_Dark_Knight': {
        'director':     ['Christopher_Nolan'],
        'release_year': ['2008'],
        'genre':        ['Action', 'Crime', 'Drama'],
        'starring':     ['Christian_Bale', 'Heath_Ledger'],
        'budget':       ['185000000'],
    }
}

entity_set = ['The_Dark_Knight']
top_k_rels  = [r for r, _ in scored[:3]]  # use top-3 scored relations
triples     = build_evidence_graph(entity_set, top_k_rels, mock_kg)

print('Selected relations:', top_k_rels)
print('\nEvidence Graph:')
print(triples_to_string(triples))

## 4. Flan-T5-XL Inference (FactKG)

In [ ]:
# NOTE: Flan-T5-XL requires ~6 GB VRAM. On CPU it will be slow.
# Comment this cell out if you don't have GPU access.

from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = 'google/flan-t5-xl'
print(f'Loading {MODEL_NAME}…')
tokenizer  = T5Tokenizer.from_pretrained(MODEL_NAME)
flan_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
flan_model.eval()
print('Model loaded!')

In [ ]:
INFERENCE_PROMPT = """Based on the given evidence graph, determine whether the claim is True or False.
Answer with only 'True' or 'False'.

Evidence Graph:
{graph}

Claim: {claim}
Answer:"""

def verify_claim(claim, triples, tokenizer, model, device):
    graph_str = triples_to_string(triples)
    prompt    = INFERENCE_PROMPT.format(graph=graph_str, claim=claim)
    inputs    = tokenizer(prompt, return_tensors='pt', max_length=512,
                          truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, num_beams=2)
    return tokenizer.decode(out[0], skip_special_tokens=True)


test_claims = [
    'Christopher Nolan directed The Dark Knight.',
    'The Dark Knight was released in 1999.',
]

for tc in test_claims:
    ans = verify_claim(tc, triples, tokenizer, flan_model, device)
    print(f'Claim : {tc}')
    print(f'Answer: {ans}\n')

## 5. MetaQA 1-hop Demo

In [ ]:
from src.utils import (METAQA_RELATIONS, infer_metaqa_relation,
                        extract_metaqa_entities, score_candidates)

mock_metaqa_kb = {
    'The_Dark_Knight': {
        'directed_by':  ['Christopher_Nolan'],
        'starred_actors': ['Christian_Bale', 'Heath_Ledger', 'Gary_Oldman'],
        'release_year': ['2008'],
        'has_genre':    ['Action', 'Crime'],
        'in_language':  ['English'],
    }
}

question = 'Who directed [The Dark Knight]?'
entities = extract_metaqa_entities(question)
print(f'Question  : {question}')
print(f'Entities  : {entities}')

# Score MetaQA relations
scored_mq = score_candidates(question, METAQA_RELATIONS, scorer, encoder, str(device))
best_rel   = scored_mq[0][0]
print(f'Top relation: {best_rel} (score={scored_mq[0][1]:.4f})')

# Retrieve answer
answers = []
for ent in entities:
    if ent in mock_metaqa_kb and best_rel in mock_metaqa_kb[ent]:
        answers.extend(mock_metaqa_kb[ent][best_rel])

print(f'\nPredicted answer(s): {answers}')
print(f'Gold answer        : Christopher_Nolan')

## Results Summary

| Dataset | Method | Score |
|---------|--------|-------|
| FactKG | KG-Flan **with** Scorer | **65.60%** |
| FactKG | KG-Flan w/o Scorer | 57.80% |
| FactKG | BERT (full supervision) | 65.20% |
| MetaQA (avg) | KG-Flan | **69.47%** Hits@1 |

The MLP Relation Scorer gives **+7.80%** on FactKG and beats fine-tuned BERT — with zero task-specific fine-tuning and near-zero inference cost.